# 第13课：模型私有化部署与 Serving

本笔记本是课堂讲义。每个知识点包含：理论知识、案例代码、讲解、易错点与练习。综合练习 P1 使用课程根目录的教学运行时 [`agent_lab`](../agent_lab/README.md)。课后独立练习见 [chapter13_模型私有化部署与Serving_课后练习.ipynb](chapter13_模型私有化部署与Serving_课后练习.ipynb)。

**阶段定位**：阶段零 · 前置基座。前置基座，为后续任务提供稳定 /v1 端点

前 12 课把表洗干净、把函数写对。从本课起，程序要**调用模型**。模型不是函数：同一输入可能换措辞，必须用协议和 Schema 把它关进笼子。

## 学习目标

1. 说明私有化部署要把模型收成稳定的推理服务，而不是在笔记本里临时 print。
2. 指出 OpenAI 兼容端点至少包含 `/v1/models` 与 `/v1/chat/completions`。
3. 使用 Pydantic `HealthReport` 做自动化探活，记录延迟与 JSON 约束是否成功。
4. 在未配置真实 GPU 服务时，能解释 Fake 端点为何仍能完成本课验收。

## 学习知识点

| 端点 | 探活 | 约束输出 |
| --- | --- | --- |
| Base URL 指向 /v1 | 列出 model id | response_format=json_object |
| Key 与超时写在配置里 | 记录 latency_ms | Pydantic 再校验 |
| Fake 与 Http 同一形状 | ok 与 json_ok 分开 | ping / serving 字段 |

## 基础回顾与案例提问

1. **R.1** 若业务代码里写死 `http://127.0.0.1:11434/v1`，换到 vLLM 的 8000 端口时要改几处？配置项应该叫什么？
2. **R.2** `/v1/models` 成功是否等于聊天也能返回合法 JSON？还缺哪一步？
3. **R.3** 模型返回 `{"ping":"ok"}` 但漏了 `serving` 时，`HealthReport.json_ok` 应是 True 还是 False？

本课不讲 CUDA、量化训练或多卡调度。真实 Ollama/vLLM 只作为可选对接。

使用 Python 3；需要 `pydantic`。从本课文件夹启动内核。本课不要求 GPU，也不强制安装 `langgraph` / `openai`。未配置私有化端点时，`get_client()` 返回进程内 Fake。不要使用 pandas。综合练习不要抄 `experiment.py` 的整段答案，按题面逐步完成。


In [ ]:
# R.1–R.3: Write and verify your predictions here.


In [ ]:
import sys
from pathlib import Path

COURSE = Path.cwd().resolve()
if COURSE.name.startswith("第"):
    COURSE = COURSE.parent
if str(COURSE) not in sys.path:
    sys.path.insert(0, str(COURSE))
print("已加入路径:", COURSE)
print("请从本课文件夹启动内核。未配置 OPENAI_BASE_URL 时使用教学 Fake 端点，不要求 GPU。")


## 1. 为什么要私有化 Serving

### 理论知识

**Serving 的意思是：模型作为服务常驻，业务只认 URL。** 数据可以不出域，接口形状要对齐，这样后面的 Agent 不用关心背后是 Qwen 还是 DeepSeek。

### 案例：对比“直接在脚本里假想有模型”和“先探活再调用”


In [ ]:
print("业务不要写: model.predict(prompt)")
print("业务要写: POST {BASE_URL}/chat/completions")
print("验收看两件事: 列表里有模型, 回复能通过 Schema")


### 讲解

把模型当成会失败的网络依赖：超时、空列表、非 JSON 都可能发生。探活脚本的职责是在 Agent 启动前把这些失败变成结构化报告。

### 易错点与练习

1. **K1.1** Serving 和“在 Jupyter 里 import 一个本地权重文件”有何不同？
2. **K1.2** 为什么实训产出要写“稳定推理服务”，而不是“能聊天”？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 2. OpenAI 兼容的 /v1

### 理论知识

**兼容不是营销词，是两套路径。** 列表用 GET `/models`，对话用 POST `/chat/completions`。请求体里有 `model`、`messages`；JSON 约束常加 `response_format`。

### 案例：看 FakeOpenAI 的返回形状


In [ ]:
from agent_lab.mock_openai import FakeOpenAI
client = FakeOpenAI()
print(client.base_url)
print(client.models.list()["data"])
resp = client.chat.completions.create(
    model="qwen2.5",
    messages=[{"role": "user", "content": "健康探活"}],
    response_format={"type": "json_object"},
)
print(resp["choices"][0]["message"]["content"])


### 讲解

`choices[0].message.content` 仍是字符串。兼容端点只保证外层信封，不保证业务字段合法，所以还要 Pydantic。

### 易错点与练习

1. **K2.1** 若 Base URL 写成 `http://127.0.0.1:11434` 而服务实际挂在 `/v1`，列表请求会打到哪？
2. **K2.2** `messages` 为什么是列表而不是一个长字符串？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 3. Fake 客户端与 Http 客户端

### 理论知识

**同一套调用形状，两种实现。** `get_client()` 读环境变量 `OPENAI_BASE_URL`；没有则 Fake；有但连不上则 fake-fallback。

### 案例：打印当前端点来源


In [ ]:
from agent_lab.mock_openai import get_client
client, source = get_client()
print(source, getattr(client, "base_url", None))


### 讲解

课堂默认 `source == 'fake'` 就算达标。连上真实 Ollama 时 source 变为 `http`。不要为了“看起来更真”而在没服务时强行 Http。

### 易错点与练习

1. **K3.1** 为什么教学要保留 Fake，而不是强制每人安装 7B 模型？
2. **K3.2** Key 写成 `sk-local` 在内网私有化里通常扮演什么角色？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 4. Pydantic 探活报告

### 理论知识

**探活结果也是一份 Schema。** `HealthReport` 记录 ok、模型 id、延迟、json_ok、detail、source。自动化脚本只认字段，不认 print 的心情。

### 案例：调用 probe()


In [ ]:
from agent_lab.health import probe
report = probe()
print(report.model_dump())


### 讲解

`ok` 需要“有模型”且“JSON 约束成功”。只有列表成功但聊天胡言，json_ok 为 False，整次探活不算过。

### 易错点与练习

1. **K4.1** latency_ms 在 Fake 下接近 0 说明了什么？能否用它判断真实 GPU 是否够快？
2. **K4.2** detail 里为什么要同时保留原始字符串和校验后的 JSON？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 5. JSON 约束输出

### 理论知识

**约束分两层。** 第一层告诉模型“只输出 JSON”；第二层用 `ProbePayload` 检查字段类型与取值。

### 案例：手动校验一段回复


In [ ]:
import json
from agent_lab.health import ProbePayload
raw = '{"ping":"ok","serving":true}'
print(ProbePayload.model_validate(json.loads(raw)))


### 讲解

模型偶尔会包 markdown 代码块或加解说。探活脚本必须把这种失败写进 detail，而不是让 `json.loads` 把整个 Agent 拉倒。

### 易错点与练习

1. **K5.1** 若 raw 是 `好的，服务正常`，应记 json_ok=False 还是手写 True？
2. **K5.2** `serving` 必须是布尔值，写成字符串 `"true"` 能否通过？先预测再验证。

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 6. 超时策略

### 理论知识

**超时是配置，不是运气。** `timeout_s` 传到 Http 客户端。内网小模型常见 8 秒；冷启动可能更长。课堂 Fake 不消耗等待。

### 案例：看 probe 的参数


In [ ]:
from agent_lab.health import probe
print(probe(timeout_s=5.0).ok)


### 讲解

把超时写进 OpenClawConfig 后，第 15 课不用再改探活代码。这就是阶段零要先封装 Serving 的原因。

### 易错点与练习

1. **K6.1** 超时过短会出现什么用户可见现象？
2. **K6.2** 为什么不建议在每次聊天时写不同的 timeout 魔法数？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 7. 可选：对接 Ollama / vLLM

### 理论知识

**真实服务只改环境，不改业务。** Ollama 默认 `http://127.0.0.1:11434/v1`；vLLM 的 OpenAI 入口常见 8000 端口。样例见 `serving.env.example`。

### 案例：读取环境变量（没有也不报错）


In [ ]:
import os
print("OPENAI_BASE_URL=", os.environ.get("OPENAI_BASE_URL") or "(未设置，使用 Fake)")


### 讲解

助教验收以 `python3 experiment.py` 为准。你本机有 GPU 可以加分演示，但不能让没 GPU 的同学无法交作业。

### 易错点与练习

1. **K7.1** vLLM 与 Ollama 对 `/v1` 的职责有何相同之处？
2. **K7.2** 为什么作业禁止把真实 Key 贴进 ipynb？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 8. 本课验收口径

### 理论知识

**过关句子是 `本课验收通过`。** 断言：`report.ok`、`report.json_ok`、模型列表含 `qwen2.5`。

### 案例：对照验收字段


In [ ]:
from agent_lab.health import probe
r = probe()
print("source", r.source)
print("models", r.model_ids)
print("json_ok", r.json_ok, r.detail)


### 讲解

后续每一课的 Client 都经这里取端点。Serving 不稳，后面的 Tool / RAG 全是噪音。

### 易错点与练习

1. **K8.1** 若有人把 qwen2.5 改成别的默认 id，experiment.py 会在哪一条断言失败？
2. **K8.2** json_ok 与 ok 同时打印的好处是什么？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 综合练习：探活一条龙

按 P1.1 → P1.2 → P1.3 顺序完成。每步都要能独立看出你做了什么。


### P1.1　准备客户端

从本课文件夹启动内核，加入课程根目录到 `sys.path`。调用 `get_client()`，打印 `source` 与 `base_url`。


In [ ]:
# P1.1: get_client and print source / base_url.


### P1.2　列出模型并聊天

列出 model id。向聊天接口发送“健康探活”，`response_format` 设为 `json_object`，打印原始 content。


In [ ]:
# P1.2: list models and create a JSON completion.


### P1.3　写成 HealthReport

调用 `probe()`（或自己构造同等字段），确认 `ok` 与 `json_ok`。不要把 True 写死在代码里。


In [ ]:
# P1.3: probe() and confirm ok / json_ok.


课后请打开 [chapter13_模型私有化部署与Serving_课后练习.ipynb](chapter13_模型私有化部署与Serving_课后练习.ipynb)。P1 自己写探活字段说明，P2 制造一次 JSON 失败并记录，P3 选做阅读环境变量样例。
